In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
annual_value_adjusted = pd.read_csv("data/outputs/annual_ca_port",
                                    index_col =0, parse_dates = [1])
quarterly_value_adjusted = pd.read_csv('data/outputs/quarter_ca_port',
                                    index_col =0, parse_dates = [1])
band_value_adjusted = pd.read_csv("data/outputs/quarter_band_ca_port",
                                    index_col =0, parse_dates = [1])
asset_data = pd.read_csv("data/processed/daily_returns.csv",
                         parse_dates = [0])


In [3]:
annual_weights = pd.read_csv("data/outputs/annually_rebalanced_weights", 
                             index_col = 0, parse_dates = [1])
quarterly_weights = pd.read_csv("data/outputs/quarterly_rebalanced_weights", 
                                index_col = 0, parse_dates = [1])
band_weights = pd.read_csv("data/outputs/quarterly_rebalanced_byband_weights", 
                           index_col = 0, parse_dates = [1])
daily_returns = pd.read_csv("data/processed/daily_returns.csv", parse_dates = [0])

In [4]:
asset_data = daily_returns[daily_returns['Date'].isin(annual_value_adjusted['Date'])].reset_index().drop(columns=['index'])

In [5]:
def daily_asset_contributions(weights_df, asset_df):

    starting_weights = weights_df[weights_df['weight_type'] == 'starting']

    start_dates = weights_df.loc[weights_df['weight_type'] == 'starting', 'Date']

    end_dates = weights_df.loc[weights_df['weight_type'] == 'ending', 'Date']

    returns_list = []

    for row in range(len(start_dates)):

        start = start_dates.iloc[row]
        
        end = end_dates.iloc[row]

        period_data = asset_df[asset_df['Date'].between(start, end)]

        weights = starting_weights[starting_weights['Date'] == start].drop(columns = ['Date','weight_type'])

        for row in range(len(period_data)):

            returns_by_asset = weights.iloc[0] * period_data.iloc[row]

            returns_by_asset['Date'] = period_data['Date'].iloc[row]

            returns_list.append(returns_by_asset)

    return pd.DataFrame(returns_list).set_index('Date')


In [6]:

annual_portfolio_asset_contributions = daily_asset_contributions(annual_weights, asset_data)
quarterly_portfolio_asset_contributions = daily_asset_contributions(quarterly_weights, asset_data)
banded_portfolio_asset_contributions = daily_asset_contributions(band_weights, asset_data)

In [16]:
def asset_contributions_to_returns(portfolio_df, asset_contribution_df):
    weights_sums = []
    total_return = portfolio_df['portfolio_return'].sum()

    for col in asset_contribution_df.columns:
        name = asset_contribution_df[col].name
        col_sum = asset_contribution_df[col].sum()
        percent_cont = col_sum / total_return
        weights_sums.append({
            'asset' : name, 
            'arethmatic_total_return' : col_sum,
            'percent_contribution' : percent_cont
        })

    return pd.DataFrame(weights_sums).set_index('asset')


In [17]:
asset_contribution_table_annual = asset_contributions_to_returns(annual_value_adjusted, annual_portfolio_asset_contributions)
asset_contribution_table_quarterly = asset_contributions_to_returns(quarterly_value_adjusted, quarterly_portfolio_asset_contributions)
asset_contribution_table_band = asset_contributions_to_returns(band_value_adjusted, banded_portfolio_asset_contributions)

In [24]:
portfolio_groups = {
    "Dom_Eq": ['SPY', 'QQQ', 'IWM'],
    "Intl_Eq": ['EFA', 'EEM'],
    "Fixed_Inc": ['AGG', 'TLT', 'LQD'],
    "Alt": ['GLD', 'VNQ'],
    "Factor": ['MTUM', 'VLUE', 'QUAL', 'USMV']
}